<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_08_model_tuning/stage_08_02_xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_08_02 - Tuning - XGBoost**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [ ]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-21 21:36:03,404 | INFO | Environment initialized


## **2. Acceso a drive**

In [ ]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

Mounted at /content/drive


2026-04-21 21:36:24,283 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [ ]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
  "t2_p40_h30",
  "t2_p40_h60",
  "t2_p50_h30",
]

# Tamaños de ventana
WINDOW_SIZES = [30]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
        "ema_60",
        "roc_60",
        "roc_30",
        "stoch_k_30",
        "mom_5",
        "atr_norm_10",
        "macd"
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-21 21:36:25,599 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-21 21:36:25,600 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-21 21:36:25,601 | INFO | Configuración de experimento cargada
2026-04-21 21:36:25,602 | INFO | Targets: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
2026-04-21 21:36:25,602 | INFO | Window sizes: [30]


In [ ]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_mnq_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:10]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[30]["t2_p40_h30"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-21 21:36:25,613 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-21 21:36:26,518 | INFO | Windows OK      : 9
2026-04-21 21:36:26,518 | INFO | Windows missing : 0
2026-04-21 21:36:26,519 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl
2026-04-21 21:36:26,520 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [ ]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [ ]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [ ]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_p40_h30'
        - 't2_p40_h60'
        - 't2_p50_h30'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon_str = target.split("_")[-1]   # ej: "h30"
        horizon = int(horizon_str.replace("h", ""))
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }

### **4.4. Creación de bundles T2**

In [ ]:
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundles : dict
        bundles[target] -> bundle dict
    """

    bundles = {}

    # --------------------------
    # Construcción
    # --------------------------
    for target in targets:
        bundles[target] = load_windows_and_scaler(
            window_size=window_size,
            target=target,
            windows_paths=windows_paths,
            scaler_path=scaler_path,
        )

    # --------------------------
    # Verificación rápida
    # --------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    for target in targets:
        b = bundles[target]

        print(f"\nTARGET: {target}")
        print("Train :", b["train"]["X"].shape, b["train"]["y"].shape)
        print("Valid :", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print("Test  :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print("Scaler:", type(b["scaler"]).__name__)

    return bundles

In [ ]:
bundles_L30 = create_bundles(window_size=30)

2026-04-21 21:36:27,744 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-21 21:36:27,745 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-21 21:36:28,464 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-21 21:36:28,464 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-21 21:36:29,111 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-21 21:36:29,112 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-21 21:36:30,886 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-21 21:36:30,887 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-21 21:36:33,288 | INFO | Loaded: windows_t2_p40_h60_train.npz
2026-04-21 21:36:33,289 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-21 21:36:34,196 | INFO | Loaded: windows_t2_p40_h60_valid.npz
2026-04-21 21:36:34,197 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-21 21:36:35,040 | INFO | Loaded: windows_t2_p4


WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p40_h60
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler


Como acceder a las ventanas X e y:

```python
bundle_p40_h30 = bundles_L30["t2_p40_h30"]

X_train = bundle_p40_h30["train"]["X"]
y_train = bundle_p40_h30["train"]["y"]

X_valid = bundle_p40_h30["valid"]["X"]
y_valid = bundle_p40_h30["valid"]["y"]

X_test = bundle_p40_h30["test"]["X"]
y_test = bundle_p40_h30["test"]["y"]

scaler = bundle_p40_h30["scaler"]

print("Target :", bundle_p40_h30["target"])
print("Horizon:", bundle_p40_h30["horizon"])
print("Train  :", X_train.shape, y_train.shape)
print("Valid  :", X_valid.shape, y_valid.shape)
print("Test   :", X_test.shape, y_test.shape)
print("Scaler :", type(scaler).__name__)
```




In [ ]:
bundles_L30

{'t2_p40_h30': {'window_size': 30,
  'target': 't2_p40_h30',
  'horizon': 30,
  'paths': {'train': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_train.npz',
   'valid': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_valid.npz',
   'test': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_test.npz',
   'scaler': '/content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl'},
  'scaler': StandardScaler(),
  'train': {'X': array([[[-0.11680385, -0.06583842, -0.2238865 , ..., -0.04129868,
            -1.507039  , -0.02778307],
           [ 0.0909589 ,  0.05818945, -0.07248875, ...,  0.3872164 ,
            -1.3723946 ,  0.08631181],
           [-0.18229802, -0.13053213, -0.21077001, ..., -0.1151728 ,
            -1.2194692 ,  0.02307655],
           ...,
           [ 0.40151003,  0.28442496,  0.59289074, ...,  0.47497907,
            -1.0690751 , -0.19294474],
         

### **4.5. Preparación de inputs según el tipo de modelo**

In [ ]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

Estos sanity checks sirven para verificar, antes de entrenar, que los datos cargados tengan la estructura correcta y no vengan con errores silenciosos.

En concreto, comprueban que:

- X tenga el formato esperado: 3D (n, seq_len, n_features) o 2D (n, d_flat)
- y tenga forma válida para clasificación seq2one
- X e y tengan la misma cantidad de muestras
- no haya NaN ni inf
- las dimensiones sean consistentes entre train, valid y test
- exista más de una clase en y

Nos conviene tenerlos, porque ayudan a detectar errores de shape o de datos antes de llegar al entrenamiento.

In [ ]:
from __future__ import annotations

from typing import Any, Optional, Tuple, Dict, Mapping
import numpy as np


# ============================================================
# SANITY CHECKS PARA DATASETS SEQ2ONE (T2)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía
    ni contenga NaN/inf.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(
    y: np.ndarray,
    *,
    name: str = "y",
    allow_seq_inputs_take_last: bool = False,
) -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)
    - (n, seq_len)       si allow_seq_inputs_take_last=True
    - (n, seq_len, 1)    si allow_seq_inputs_take_last=True
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    if allow_seq_inputs_take_last:
        if y.ndim == 2 and y.shape[1] > 1:
            return y[:, -1]

        if y.ndim == 3 and y.shape[2] == 1:
            return y[:, -1, 0]

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1)"
        f"{' o secuencial si allow_seq_inputs_take_last=True' if allow_seq_inputs_take_last else ''}. "
        f"Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )


def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)
      - y secuencial, opcionalmente, tomando el último valor

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: no aplica directamente.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D.
    allow_seq_inputs_take_last:
        Si y viene como secuencia, toma el último valor.
    """
    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    y = _normalize_y_seq2one(
        y,
        name=f"y[{split_name}]",
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
    )

    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info


def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_p40_h30",
      "horizon": 30,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Si no se pasan expected_*, usa TRAIN como referencia.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_bundles_seq2one(
    bundles: Mapping[str, Dict[str, Any]],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para todos los bundles de un diccionario:

    bundles[target] -> bundle
    """
    results = {}

    for target, bundle in bundles.items():
        results[target] = run_sanity_checks_for_bundle_seq2one(
            bundle,
            tag=target,
            verbose=verbose,
        )

    return results

In [ ]:
sanity_results = run_sanity_checks_all_bundles_seq2one(
    bundles_L30,
    verbose=True,
)

[sanity_check_seq2one] train_t2_p40_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h30 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h30 | target=t2_p40_h30 | horizon=30
[sanity_check_seq2one] train_t2_p40_h60 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h60 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h60 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h60 | target=t2_p40_h60 | horizon=60
[sanity_check_seq2one] train_t2_p50_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p50_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | c

## **6. Módulo de métricas T2**

In [ ]:
# ================================
# Setup para importar módulos del proyecto
# ================================

import sys
import importlib

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

# asegurar que metrics es paquete
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

from metrics.classification_probabilities import (
    compute_probabilistic_outputs,
    apply_decision_rule,
)

print("Módulos importados correctamente")

Módulos importados correctamente


In [ ]:
# ================================
# Utilidades: outputs -> DataFrame
# ================================

from __future__ import annotations

import pandas as pd
from typing import Any


def classification_metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output de compute_classification_metrics(...)
    en una fila de DataFrame.

    Usa metrics_to_flat_dict(...) para aplanar la salida
    del módulo classification_metrics y luego agrega metadata
    del experimento.
    """
    flat_metrics = metrics_to_flat_dict(metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_metrics,
    }

    return pd.DataFrame([row])


def _flatten_dict(
    d: dict[str, Any],
    *,
    parent_key: str = "",
    sep: str = "_",
) -> dict[str, Any]:
    """
    Aplana un diccionario arbitrario de forma recursiva.

    Ejemplo:
    {"a": {"b": 1}} -> {"a_b": 1}
    """
    items: dict[str, Any] = {}

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)

        if isinstance(v, dict):
            items.update(_flatten_dict(v, parent_key=new_key, sep=sep))
        else:
            items[new_key] = v

    return items


def probabilities_metrics_to_df(
    prob_metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output del módulo classification_probabilities
    en una fila de DataFrame.

    Como la estructura puede variar según la implementación,
    se aplana recursivamente el diccionario y luego se agrega
    metadata del experimento.
    """
    flat_prob_metrics = _flatten_dict(prob_metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_prob_metrics,
    }

    return pd.DataFrame([row])


logger.info("Utilidades de exportación a DataFrame cargadas")

2026-04-21 21:36:41,238 | INFO | Utilidades de exportación a DataFrame cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [ ]:
# ================================
# Persistencia de métricas de clasificación
# ================================

from pathlib import Path
import pandas as pd


def load_classification_metrics_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen métricas previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path


# ================================
# Persistencia de probabilidades / decisión
# ================================

def load_classification_probabilities_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_probabilities",
) -> pd.DataFrame:
    """
    Carga resultados probabilísticos si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando probabilidades desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen probabilidades previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_probabilities(
    df_probabilities: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_probabilities",
) -> Path:
    """
    Guarda un DataFrame de probabilidades / decisión en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"
    df_probabilities.to_parquet(out_path, index=False)

    logger.info(f"Probabilidades guardadas en: {out_path}")

    return out_path

Ejemplo de uso con logistic regression:

```python
df_metrics_all = load_classification_metrics_if_exists(
    model_name="logistic_regression",
    split="valid",
)

df_probabilities_all = load_classification_probabilities_if_exists(
    model_name="logistic_regression",
    split="valid",
)
```

Guardar:
```python
save_classification_metrics(
    df_metrics_all,
    model_name="logistic_regression",
    split="valid",
)

save_classification_probabilities(
    df_probabilities_all,
    model_name="logistic_regression",
    split="valid",
)
```

## **8. Gestión de dispositivo y memoria**

In [ ]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [ ]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-21 21:36:45,345 | INFO | Seeds fijadas en 42


# **10. Entrenamiento de modelo XGBoost**

## **10.1. Función unitaria por bundle**

In [ ]:
from xgboost import XGBClassifier
import numpy as np


def run_xgboost_for_bundle_seq2one(
    bundle,
    *,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    min_child_weight=1,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    input_mode="2d_flat",
    class_weight=None,
    tree_method="hist",
    device="cuda",   # "cuda" o "cpu"
    verbose=False,
):
    """
    Ejecuta XGBoost para un bundle seq2one.
    Evalúa SOLO sobre VALID.

    Usa labels arbitrarias (ej. [-1, 0, 1]) mediante codificación interna.
    """

    # =========================
    # 1. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    target = bundle.get("target")
    horizon = bundle.get("horizon")
    window_size = bundle.get("window_size")

    # =========================
    # 2. PREPARAR INPUT
    # =========================
    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)

    # =========================
    # 3. CODIFICAR LABELS
    # =========================
    classes_ = np.sort(np.unique(y_train))
    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int32)
    num_class = len(classes_)

    # =========================
    # 4. SAMPLE WEIGHTS
    # =========================
    sample_weight = None

    if class_weight is None:
        sample_weight = None

    elif class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_class)
        total = counts.sum()

        weights_by_idx = {
            idx: total / (num_class * count)
            for idx, count in enumerate(counts)
        }

        sample_weight = np.array(
            [weights_by_idx[idx] for idx in y_train_enc],
            dtype=np.float32,
        )

    elif isinstance(class_weight, dict):
        weights_by_idx = {
            class_to_idx[cls]: weight
            for cls, weight in class_weight.items()
            if cls in class_to_idx
        }

        sample_weight = np.array(
            [weights_by_idx.get(idx, 1.0) for idx in y_train_enc],
            dtype=np.float32,
        )

    else:
        raise ValueError("class_weight debe ser None, 'balanced' o dict")

    # =========================
    # 5. MODELO
    # =========================
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=num_class,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=random_state,
        n_jobs=n_jobs,
        eval_metric="mlogloss",
        tree_method=tree_method,
        device=device,
        verbosity=1 if verbose else 0,
    )

    # =========================
    # 6. TRAIN
    # =========================
    model.fit(
        X_train_model,
        y_train_enc,
        sample_weight=sample_weight,
    )

    # =========================
    # 7. PREDICT (VALID)
    # =========================
    y_pred_valid_enc = model.predict(X_valid_model)
    y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])

    y_proba_valid = model.predict_proba(X_valid_model)

    if verbose:
        print(
            f"[XGBOOST] target={target} | horizon={horizon} | "
            f"window_size={window_size} | "
            f"device={device} | tree_method={tree_method} | "
            f"X_train={X_train_model.shape} | X_valid={X_valid_model.shape}"
        )

    return {
        "model_name": "xgboost",
        "target": target,
        "horizon": horizon,
        "window_size": window_size,
        "input_mode": input_mode,
        "class_weight": class_weight,
        "tree_method": tree_method,
        "device": device,
        "model": model,
        "classes_": classes_,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "sample_weight": sample_weight,
        "y_valid": y_valid,
        "y_pred_valid": y_pred_valid,
        "y_proba_valid": y_proba_valid,
    }

## **10.2. Función de evaluación sobre uno o más bundles**

In [ ]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd


def eval_xgboost_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    model_name: str = "xgboost",
    n_estimators: int = 100,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,
    min_child_weight: float = 1.0,
    gamma: float = 0.0,
    reg_alpha: float = 0.0,
    reg_lambda: float = 1.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
    tree_method: str = "hist",
    device: str = "cuda",
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
    verbose: bool = False,
) -> Dict[str, pd.DataFrame]:
    """
    Evalúa XGBoost para uno o varios bundles seq2one
    usando SOLO el split VALID.

    Retorna
    -------
    dict con:
    - "metrics": DataFrame de métricas de clasificación
    - "probabilities": DataFrame de métricas probabilísticas / decisión
    """

    # --------------------------------------------------
    # 1) Normalizar entrada
    # --------------------------------------------------
    if isinstance(bundles, dict):
        if "train" in bundles and "valid" in bundles:
            bundles_list: List[Dict[str, Any]] = [bundles]
        else:
            bundles_list = list(bundles.values())
    else:
        bundles_list = list(bundles)

    metrics_rows = []
    probabilities_rows = []

    # --------------------------------------------------
    # 2) Iterar por bundles
    # --------------------------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"model={model_name} | "
                f"class_weight={class_weight}"
            )

        # ----------------------------------------------
        # 3) Entrenar + predecir SOLO VALID
        # ----------------------------------------------
        preds = run_xgboost_for_bundle_seq2one(
            bundle,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            min_child_weight=min_child_weight,
            gamma=gamma,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
            tree_method=tree_method,
            device=device,
            verbose=False,
        )

        y_true = preds["y_valid"]
        y_pred = preds["y_pred_valid"]
        y_proba = preds["y_proba_valid"]

        # ----------------------------------------------
        # 4) Métricas de clasificación
        # ----------------------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split="valid",
            target=target,
            labels=[-1, 0, 1],
        )

        df_metrics_row = classification_metrics_to_df(
            metrics,
            model=model_name,
            split="valid",
            window_size=window_size,
            target=target,
            horizon=horizon,
        )

        if class_weight == "balanced":
            class_weight_mode = "balanced"
        elif class_weight is None:
            class_weight_mode = "none"
        else:
            class_weight_mode = "custom"

        df_metrics_row["class_weight_mode"] = class_weight_mode
        df_metrics_row["input_mode"] = input_mode
        df_metrics_row["n_estimators"] = n_estimators
        df_metrics_row["max_depth"] = max_depth
        df_metrics_row["learning_rate"] = learning_rate
        df_metrics_row["subsample"] = subsample
        df_metrics_row["colsample_bytree"] = colsample_bytree
        df_metrics_row["min_child_weight"] = min_child_weight
        df_metrics_row["gamma"] = gamma
        df_metrics_row["reg_alpha"] = reg_alpha
        df_metrics_row["reg_lambda"] = reg_lambda
        df_metrics_row["tree_method"] = tree_method
        df_metrics_row["device"] = device
        df_metrics_row["n_jobs"] = n_jobs

        metrics_rows.append(df_metrics_row)

        # ----------------------------------------------
        # 5) Outputs probabilísticos
        # ----------------------------------------------
        proba_df = compute_probabilistic_outputs(
            y_proba=y_proba,
            class_labels=[-1, 0, 1],
            y_true=y_true,
        )

        decision_df = apply_decision_rule(
            proba_df,
            long_class=1,
            short_class=-1,
            long_threshold=prob_threshold_long,
            short_threshold=prob_threshold_short,
        )

        overlap_cols = [c for c in decision_df.columns if c in proba_df.columns]
        if overlap_cols:
            decision_df = decision_df.drop(columns=overlap_cols)

        df_prob = pd.concat(
            [proba_df.reset_index(drop=True), decision_df.reset_index(drop=True)],
            axis=1,
        )

        df_prob["model"] = model_name
        df_prob["split"] = "valid"
        df_prob["window_size"] = window_size
        df_prob["target"] = target
        df_prob["horizon"] = horizon
        df_prob["class_weight_mode"] = class_weight_mode
        df_prob["input_mode"] = input_mode
        df_prob["n_estimators"] = n_estimators
        df_prob["max_depth"] = max_depth
        df_prob["learning_rate"] = learning_rate
        df_prob["subsample"] = subsample
        df_prob["colsample_bytree"] = colsample_bytree
        df_prob["min_child_weight"] = min_child_weight
        df_prob["gamma"] = gamma
        df_prob["reg_alpha"] = reg_alpha
        df_prob["reg_lambda"] = reg_lambda
        df_prob["tree_method"] = tree_method
        df_prob["device"] = device
        df_prob["n_jobs"] = n_jobs
        df_prob["threshold_long"] = prob_threshold_long
        df_prob["threshold_short"] = prob_threshold_short

        probabilities_rows.append(df_prob)

    # --------------------------------------------------
    # 6) Consolidar salida
    # --------------------------------------------------
    df_metrics_all = pd.concat(metrics_rows, ignore_index=True)
    df_probabilities_all = pd.concat(probabilities_rows, ignore_index=True)

    return {
        "metrics": df_metrics_all,
        "probabilities": df_probabilities_all,
    }

## **10.3. Función orquestadora por `window_size`**

5) Configuración final de experimentos anteriores
- window_size: 30
- target: t2_dir_thr_90
- n_estimators: 200
- max_depth: 3
- learning_rate: 0.03
- subsample: 0.8
- colsample_bytree: 0.8
- min_child_weight: 1–3
- gamma: 0.0
- reg_alpha: 0.0
- reg_lambda: 10.0

In [ ]:
import gc
import pandas as pd


def run_xgboost(
    window_size: int,
    *,
    targets: list[str] = TARGETS,
    verbose: bool = True,
    model_name: str = "xgboost",
    n_estimators: int = 100,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,
    min_child_weight: float = 1.0,
    gamma: float = 0.0,
    reg_alpha: float = 0.0,
    reg_lambda: float = 1.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
    tree_method: str = "hist",
    device: str = "cuda",
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
) -> dict[str, pd.DataFrame]:
    """
    Ejecuta XGBoost para una sola window_size
    sobre los targets T2 indicados.

    Evalúa SOLO sobre VALID.

    Retorna
    -------
    dict con:
    - "metrics": DataFrame consolidado de métricas de clasificación
    - "probabilities": DataFrame consolidado de métricas probabilísticas / decisión
    """

    size = int(window_size)

    bundles = None
    results = None

    model_name_effective = (
        f"{model_name}_balanced" if class_weight == "balanced" else model_name
    )

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"XGBOOST | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"targets            = {targets}")
            print(f"class_weight       = {class_weight}")
            print(f"input_mode         = {input_mode}")
            print(f"n_estimators       = {n_estimators}")
            print(f"max_depth          = {max_depth}")
            print(f"learning_rate      = {learning_rate}")
            print(f"subsample          = {subsample}")
            print(f"colsample_bytree   = {colsample_bytree}")
            print(f"min_child_weight   = {min_child_weight}")
            print(f"gamma              = {gamma}")
            print(f"reg_alpha          = {reg_alpha}")
            print(f"reg_lambda         = {reg_lambda}")
            print(f"tree_method        = {tree_method}")
            print(f"device             = {device}")
            print(f"n_jobs             = {n_jobs}")
            print(f"thr_long           = {prob_threshold_long}")
            print(f"thr_short          = {prob_threshold_short}")

        # --------------------------------------------------
        # 2) Construcción de bundles
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | n_targets={len(targets)}")

        bundles = create_bundles(
            window_size=size,
            targets=targets,
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        # --------------------------------------------------
        # 3) Evaluación SOLO VALID
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[EVAL] L{size} | split=valid | "
                f"model={model_name_effective} | class_weight={class_weight}"
            )

        results = eval_xgboost_bundles(
            bundles=bundles,
            model_name=model_name_effective,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            min_child_weight=min_child_weight,
            gamma=gamma,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
            tree_method=tree_method,
            device=device,
            prob_threshold_long=prob_threshold_long,
            prob_threshold_short=prob_threshold_short,
            verbose=verbose,
        )

        df_metrics = (
            results["metrics"]
            .sort_values(["window_size", "target", "split", "horizon", "model"])
            .reset_index(drop=True)
        )

        df_probabilities = (
            results["probabilities"]
            .sort_values(["window_size", "target", "split", "horizon", "model"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 4) Resumen final
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[DONE] L{size} | "
                f"metrics_rows={len(df_metrics)} | "
                f"probabilities_rows={len(df_probabilities)}"
            )

            print("\n[METRICS]")
            cols_metrics = [
                c for c in [
                    "window_size",
                    "split",
                    "target",
                    "model",
                    "horizon",
                    "class_weight_mode",
                    "balanced_accuracy",
                    "f1_macro",
                ] if c in df_metrics.columns
            ]
            if cols_metrics:
                print(df_metrics[cols_metrics].to_string(index=False))

            print("\n[PROBABILITIES]")
            cols_probs = [
                c for c in [
                    "window_size",
                    "split",
                    "target",
                    "model",
                    "horizon",
                    "class_weight_mode",
                    "threshold_long",
                    "threshold_short",
                ] if c in df_probabilities.columns
            ]
            if cols_probs:
                print(df_probabilities[cols_probs].to_string(index=False))

        return {
            "metrics": df_metrics,
            "probabilities": df_probabilities,
        }

    finally:
        del bundles, results
        gc.collect()

## **10.4. Ejecución final del experimento**

In [ ]:
results_xgb = run_xgboost(
    window_size=30,
    targets=["t2_p40_h30", "t2_p40_h60", "t2_p50_h30"],
    model_name="xgboost",
    n_estimators=200,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=10.0,
    random_state=SEED,
    n_jobs=-1,
    input_mode="2d_flat",
    class_weight="balanced",
    tree_method="hist",
    device="cuda",
    prob_threshold_long=0.40,
    prob_threshold_short=0.40,
    verbose=True,
)

df_metrics_xgb = results_xgb["metrics"]
df_probabilities_xgb = results_xgb["probabilities"]

2026-04-21 21:49:09,464 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-21 21:49:09,465 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-21 21:49:09,486 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-21 21:49:09,486 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-21 21:49:09,504 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-21 21:49:09,505 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-21 21:49:09,508 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-21 21:49:09,509 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-21 21:49:09,585 | INFO | Loaded: windows_t2_p40_h60_train.npz
2026-04-21 21:49:09,585 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)



XGBOOST | T2 SEQ2ONE | WINDOW_SIZE=L30
targets            = ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
class_weight       = balanced
input_mode         = 2d_flat
n_estimators       = 200
max_depth          = 3
learning_rate      = 0.03
subsample          = 0.8
colsample_bytree   = 0.8
min_child_weight   = 1
gamma              = 0.0
reg_alpha          = 0.0
reg_lambda         = 10.0
tree_method        = hist
device             = cuda
n_jobs             = -1
thr_long           = 0.4
thr_short          = 0.4

[BUILD] L30 | n_targets=3


2026-04-21 21:49:09,606 | INFO | Loaded: windows_t2_p40_h60_valid.npz
2026-04-21 21:49:09,606 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-21 21:49:09,624 | INFO | Loaded: windows_t2_p40_h60_test.npz
2026-04-21 21:49:09,625 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-21 21:49:09,629 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-21 21:49:09,629 | INFO | Bundle cargado | target=t2_p40_h60 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-21 21:49:09,708 | INFO | Loaded: windows_t2_p50_h30_train.npz
2026-04-21 21:49:09,708 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-21 21:49:09,726 | INFO | Loaded: windows_t2_p50_h30_valid.npz
2026-04-21 21:49:09,726 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-21 21:49:09,743 | INFO | Loaded: windows_t2_p50_h30_test.npz
2026-04-21 21:49:09,743 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-21 21:49:09,746 | INFO | Scaler cargado: scaler_m

Se han truncado las últimas 5000 líneas del flujo de salida.
          30 valid t2_p50_h30 xgboost_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 xgboost_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 xgboost_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 xgboost_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 xgboost_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 xgboost_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 xgboost_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 xgboost_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 xgboost_balanced       30          balanced          

## **10.5. Métricas**

In [ ]:
print('df_metrics_xgb')
df_metrics_xgb

df_metrics_xgb


,model,split,window_size,target,horizon,n_samples,accuracy,balanced_accuracy,f1_macro,f1_weighted,...,learning_rate,subsample,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda,tree_method,device,n_jobs
0,xgboost_balanced,valid,30,t2_p40_h30,30,6882,0.387823,0.405681,0.385198,0.378580,...,0.03,0.8,0.8,1,0.0,0.0,10.0,hist,cuda,-1
1,xgboost_balanced,valid,30,t2_p40_h60,60,6882,0.404098,0.398670,0.396124,0.400245,...,0.03,0.8,0.8,1,0.0,0.0,10.0,hist,cuda,-1
2,xgboost_balanced,valid,30,t2_p50_h30,30,6882,0.414560,0.408889,0.403778,0.407705,...,0.03,0.8,0.8,1,0.0,0.0,10.0,hist,cuda,-1


In [ ]:
print('df_probabilities_xgb')
df_probabilities_xgb

df_probabilities_xgb


,proba_-1,proba_0,proba_1,pred_label,confidence,y_true,is_correct,signal_raw,trade,model,...,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda,tree_method,device,n_jobs,threshold_long,threshold_short
0,0.264024,0.582266,0.153710,0,0.582266,1,False,0,False,xgboost_balanced,...,0.8,1,0.0,0.0,10.0,hist,cuda,-1,0.4,0.4
1,0.250101,0.542760,0.207139,0,0.542760,1,False,0,False,xgboost_balanced,...,0.8,1,0.0,0.0,10.0,hist,cuda,-1,0.4,0.4
2,0.241428,0.524195,0.234376,0,0.524195,1,False,0,False,xgboost_balanced,...,0.8,1,0.0,0.0,10.0,hist,cuda,-1,0.4,0.4
3,0.231874,0.542368,0.225758,0,0.542368,1,False,0,False,xgboost_balanced,...,0.8,1,0.0,0.0,10.0,hist,cuda,-1,0.4,0.4
4,0.214539,0.557100,0.228360,0,0.557100,1,False,0,False,xgboost_balanced,...,0.8,1,0.0,0.0,10.0,hist,cuda,-1,0.4,0.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20641,0.358203,0.279878,0.361919,1,0.361919,0,False,0,False,xgboost_balanced,...,0.8,1,0.0,0.0,10.0,hist,cuda,-1,0.4,0.4
20642,0.335327,0.286100,0.378572,1,0.378572,0,False,0,False,xgboost_balanced,...,0.8,1,0.0,0.0,10.0,hist,cuda,-1,0.4,0.4
20643,0.308825,0.284286,0.406889,1,0.406889,0,False,1,True,xgboost_balanced,...,0.8,1,0.0,0.0,10.0,hist,cuda,-1,0.4,0.4
20644,0.315529,0.286617,0.397855,1,0.397855,0,False,0,False,xgboost_balanced,...,0.8,1,0.0,0.0,10.0,hist,cuda,-1,0.4,0.4


### Guardado de métricas

In [ ]:
save_classification_metrics(
    df_metrics_xgb,
    model_name="xgboost",
    split="valid",
)

save_classification_probabilities(
    df_probabilities_xgb,
    model_name="xgboost",
    split="valid",
)

2026-04-21 21:50:12,760 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_metrics_xgboost_valid.parquet
2026-04-21 21:50:12,813 | INFO | Probabilidades guardadas en: /content/drive/MyDrive/neural_profit/metrics/classification_probabilities/classification_probabilities_xgboost_valid.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/classification_probabilities/classification_probabilities_xgboost_valid.parquet')

## **10.6. Análisis rápido**

In [ ]:
def analyze_model_results(
    df_metrics: pd.DataFrame,
    df_probabilities: pd.DataFrame,
) -> None:
    """
    Análisis rápido de resultados de un modelo de clasificación T2.

    Imprime:
    - ranking de métricas
    - resumen operativo
    - precisión en trades
    - conclusión automática
    """

    # =============================
    # 1) Ranking clasificación
    # =============================
    print("\n" + "="*80)
    print("RANKING CLASIFICACIÓN")
    print("="*80)

    cols_cls = [
        "target",
        "horizon",
        "accuracy",
        "balanced_accuracy",
        "balanced_accuracy_gain_vs_naive",
        "f1_macro",
        "f1_weighted",
    ]

    print(
        df_metrics[cols_cls]
        .sort_values("balanced_accuracy", ascending=False)
        .to_string(index=False)
    )

    # =============================
    # 2) Resumen operativo
    # =============================
    print("\n" + "="*80)
    print("RESUMEN OPERATIVO POR TARGET")
    print("="*80)

    summary_probs = (
        df_probabilities
        .groupby(["target", "horizon"], as_index=False)
        .agg(
            n_obs=("target", "size"),
            trade_rate=("trade", "mean"),
            confidence_mean=("confidence", "mean"),
            confidence_median=("confidence", "median"),
            pct_pred_short=("pred_label", lambda s: (s == -1).mean()),
            pct_pred_flat=("pred_label", lambda s: (s == 0).mean()),
            pct_pred_long=("pred_label", lambda s: (s == 1).mean()),
            pct_signal_short=("signal_raw", lambda s: (s == -1).mean()),
            pct_signal_flat=("signal_raw", lambda s: (s == 0).mean()),
            pct_signal_long=("signal_raw", lambda s: (s == 1).mean()),
        )
    )

    print(summary_probs.to_string(index=False))

    # =============================
    # 3) Precisión en trades
    # =============================
    print("\n" + "="*80)
    print("PRECISIÓN SOLO EN TRADES")
    print("="*80)

    trade_only = df_probabilities[df_probabilities["trade"] == True]

    if len(trade_only) == 0:
        print("No hubo trades con los thresholds actuales.")
    else:
        trade_summary = (
            trade_only
            .groupby(["target", "horizon"], as_index=False)
            .agg(
                n_trades=("trade", "size"),
                precision_trades=("is_correct", "mean"),
                confidence_mean_trade=("confidence", "mean"),
                pct_trade_short=("signal_raw", lambda s: (s == -1).mean()),
                pct_trade_long=("signal_raw", lambda s: (s == 1).mean()),
            )
        )
        print(trade_summary.to_string(index=False))

    # =============================
    # 4) Conclusión automática
    # =============================
    print("\n" + "="*80)
    print("CONCLUSIÓN RÁPIDA")
    print("="*80)

    best_target = (
        df_metrics.sort_values("balanced_accuracy", ascending=False)
        .iloc[0]["target"]
    )

    print(f"Mejor target por balanced_accuracy: {best_target}")

    avg_trade_rate = df_probabilities["trade"].mean()
    print(f"Trade rate global: {avg_trade_rate:.4f}")

    if avg_trade_rate < 0.02:
        print("Diagnóstico: el modelo está siendo muy conservador.")
    elif avg_trade_rate < 0.10:
        print("Diagnóstico: el modelo opera poco; revisar thresholds.")
    else:
        print("Diagnóstico: el modelo genera una cantidad razonable de señales.")

In [ ]:
analyze_model_results(
    df_metrics_xgb,
    df_probabilities_xgb
)


RANKING CLASIFICACIÓN
    target  horizon  accuracy  balanced_accuracy  balanced_accuracy_gain_vs_naive  f1_macro  f1_weighted
t2_p50_h30       30  0.414560           0.408889                         0.075556  0.403778     0.407705
t2_p40_h30       30  0.387823           0.405681                         0.072348  0.385198     0.378580
t2_p40_h60       60  0.404098           0.398670                         0.065336  0.396124     0.400245

RESUMEN OPERATIVO POR TARGET
    target  horizon  n_obs  trade_rate  confidence_mean  confidence_median  pct_pred_short  pct_pred_flat  pct_pred_long  pct_signal_short  pct_signal_flat  pct_signal_long
t2_p40_h30       30   6882    0.259954         0.421458           0.406334        0.347573       0.429672       0.222755          0.130485         0.740046         0.129468
t2_p40_h60       60   6882    0.282040         0.437952           0.418565        0.294972       0.414705       0.290323          0.148358         0.717960         0.133682
t2_p50_h

## **10.7. Barrido de thresholds**

In [ ]:
# =========================================
# Barrido de thresholds para XGBoost
# =========================================

import numpy as np
import pandas as pd


def sweep_thresholds(
    df_probabilities: pd.DataFrame,
    *,
    thresholds: list[float] | None = None,
) -> pd.DataFrame:
    """
    Evalúa múltiples thresholds sobre un DataFrame de probabilidades
    ya generado por un modelo.

    Requiere columnas:
    - target
    - proba_-1
    - proba_1
    - y_true

    Retorna
    -------
    pd.DataFrame con:
    - target
    - threshold
    - trade_rate
    - n_trades
    - precision_trades
    """

    if thresholds is None:
        thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55]

    results = []

    for thr in thresholds:
        df = df_probabilities.copy()

        # -----------------------------
        # Regla de decisión con threshold
        # -----------------------------
        df["trade_thr"] = (
            (df["proba_1"] >= thr) | (df["proba_-1"] >= thr)
        )

        df["signal_thr"] = 0
        df.loc[df["proba_1"] >= thr, "signal_thr"] = 1
        df.loc[df["proba_-1"] >= thr, "signal_thr"] = -1

        df["correct_thr"] = df["signal_thr"] == df["y_true"]

        # -----------------------------
        # Resumen por target
        # -----------------------------
        summary = (
            df.groupby("target", as_index=False)
            .agg(
                n_obs=("target", "size"),
                n_trades=("trade_thr", "sum"),
                trade_rate=("trade_thr", "mean"),
            )
        )

        # precisión solo en trades
        precision_rows = []
        for target in summary["target"]:
            sub = df[(df["target"] == target) & (df["trade_thr"] == True)]

            if len(sub) > 0:
                precision_trades = sub["correct_thr"].mean()
            else:
                precision_trades = np.nan

            precision_rows.append(precision_trades)

        summary["precision_trades"] = precision_rows
        summary["threshold"] = thr

        results.append(summary)

    df_thresholds = (
        pd.concat(results, ignore_index=True)
        .sort_values(["target", "threshold"])
        .reset_index(drop=True)
    )

    return df_thresholds

In [ ]:
df_thresholds_xgb = sweep_thresholds(df_probabilities_xgb)

print(df_thresholds_xgb.to_string(index=False))

    target  n_obs  n_trades  trade_rate  precision_trades  threshold
t2_p40_h30   6882      5482    0.796571          0.369391       0.30
t2_p40_h30   6882      3958    0.575124          0.393633       0.35
t2_p40_h30   6882      1789    0.259954          0.434880       0.40
t2_p40_h30   6882       472    0.068585          0.425847       0.45
t2_p40_h30   6882       188    0.027318          0.500000       0.50
t2_p40_h30   6882        53    0.007701          0.490566       0.55
t2_p40_h60   6882      5381    0.781895          0.349192       0.30
t2_p40_h60   6882      4053    0.588928          0.359980       0.35
t2_p40_h60   6882      1941    0.282040          0.358578       0.40
t2_p40_h60   6882       947    0.137605          0.327350       0.45
t2_p40_h60   6882       403    0.058559          0.292804       0.50
t2_p40_h60   6882       116    0.016856          0.310345       0.55
t2_p50_h30   6882      5375    0.781023          0.334326       0.30
t2_p50_h30   6882      3800    0.5

### **Relación entre threshold, frecuencia de operación y precisión**



Se observa de forma consistente la relación esperada entre el umbral de decisión y el comportamiento operativo del modelo. A medida que aumenta el threshold, disminuye el trade_rate y aumenta la precisión de las operaciones. Este comportamiento es claro en todos los targets evaluados, particularmente en `t2_p40_h30`, donde se pasa de una alta frecuencia de operaciones con baja precisión a una baja frecuencia con mayor precisión al incrementar el umbral.

**Comportamiento por target**

El target `t2_p40_h30` presenta el comportamiento más estable. A medida que aumenta el threshold, la precisión mejora de forma consistente, alcanzando valores entre 0.43 y 0.50, manteniendo un trade_rate razonable en niveles intermedios. Esto lo posiciona como el candidato más sólido desde el punto de vista operativo.

El target `t2_p50_h30` muestra un comportamiento similar pero más débil. En la mayoría de los thresholds presenta menor precisión que `t2_p40_h30`, y solo alcanza valores comparables en niveles altos de threshold, donde el número de operaciones es significativamente bajo. Esto indica un comportamiento más selectivo pero no necesariamente superior.

El target `t2_p40_h60` presenta un desempeño claramente inferior. La precisión se mantiene baja en todos los thresholds y no mejora al aumentar el umbral, lo que sugiere ausencia de señal útil. Este target puede considerarse descartado en esta etapa.

**Punto de equilibrio entre precisión y frecuencia**

Desde una perspectiva operativa, el threshold alrededor de 0.40 representa un punto de equilibrio razonable. En el caso de `t2_p40_h30`, permite mantener un nivel de actividad cercano al 25–26% de las observaciones con una precisión en torno a 0.43–0.44, lo que constituye un balance adecuado entre volumen y calidad de señal.

Valores más altos de threshold, como 0.50, incrementan la precisión hasta aproximadamente 0.50, pero reducen drásticamente la cantidad de operaciones, lo que limita su utilidad práctica en esta etapa.

**Calidad de las probabilidades del modelo**

El modelo XGBoost responde correctamente al ajuste de thresholds, lo que indica que la estructura de probabilidades es informativa. Sin embargo, no genera probabilidades extremas con frecuencia, ya que es necesario utilizar thresholds elevados para alcanzar niveles de precisión altos. Esto sugiere que las probabilidades no están fuertemente calibradas y que la señal, aunque consistente, es relativamente débil.

Este comportamiento es típico en problemas financieros ruidosos, donde el modelo captura patrones reales pero con baja confianza individual.

**Observación general en el contexto de comparación de modelos**

En esta etapa, centrada en la comparación entre modelos, se observa que XGBoost:

* logra el mejor desempeño en términos de métricas de clasificación
* mantiene un comportamiento operativo coherente bajo distintas configuraciones de threshold
* identifica en `t2_p40_h30` el target más robusto

Sin embargo, la mejora respecto a modelos más simples es moderada, especialmente en términos de precisión de trades, lo que indica que aún no hay una ventaja dominante clara.

**Conclusión de etapa**

Los resultados obtenidos confirman que XGBoost es actualmente el modelo con mejor desempeño general dentro del conjunto evaluado, y que el target `t2_p40_h30` presenta la mejor combinación de estabilidad y calidad de señal.

No obstante, dado que esta etapa corresponde únicamente a la comparación entre modelos, la selección final del modelo y configuración óptima deberá realizarse posteriormente considerando en conjunto todos los resultados obtenidos.
